# H-JEPA on Colab (T4): reproduce the paper's planning evaluation

This notebook runs steps 1 and 2 of the plan:
1. Download the released H-JEPA checkpoints and the fixed evaluation tasks.
2. Run the repository's own planning evaluation (`h_jepa/eval.py`) on a GPU, then compare the success rates with the paper.

No training happens here. The evaluation needs only a checkpoint, its `normalizer.pt` and the task file.

**Before running:** choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

**Resumable:** with `USE_DRIVE = True`, checkpoints and results are saved to Google Drive. The evaluation writes one result file per task, so after a disconnect you can rerun all cells and it skips the finished tasks.

Pinned code version: `kevinghst/H-JEPA` at commit `97054ec` (2026-10-09).

## 0. Settings
Edit this cell only. The defaults run the smallest useful reproduction: FourRoom, three-level H-JEPA, seed 42.

In [ ]:
# ---- What to evaluate -------------------------------------------------------
# Each run is (env, model, seed).
#   env:   'fourroom' | 'ant' | 'cube' | 'pusht'
#   model: 'lewm' | 'hjepa_l2' | 'hjepa_l3' | 'hjepa_l4' | 'hwm_l2' | 'hwm_l3' | 'hwm_l4'
#   seed:  42 | 43 | 44   (the paper pairs the planner seed with the model seed)
RUNS = [
    ('fourroom', 'hjepa_l3', 42),
    # ('fourroom', 'lewm', 42),      # flat baseline, the paper's 40.7%
    # ('ant', 'hjepa_l3', 42),       # the headline 73.3% result (MuJoCo, slower)
]

SMOKE_TASKS = 2          # quick end-to-end check on the first N tasks before the full run (0 = skip)
RUN_FULL_EVAL = True     # False: stop after the smoke test
USE_DRIVE = True         # keep checkpoints/results on Google Drive (survives disconnects)

REPO_URL = 'https://github.com/kevinghst/H-JEPA.git'
REPO_COMMIT = '97054ecaff83e13839195ec4b92d74c08a0127c5'

## 1. Check the GPU

In [ ]:
import subprocess, sys, os, shlex, time, json, pathlib

def run(cmd, cwd=None, env=None, check=True):
    # Run a shell command, stream its output live, raise if it fails.
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'command failed with exit code {p.returncode}: {cmd}')
    return p.returncode

out = subprocess.run('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader',
                     shell=True, capture_output=True, text=True)
assert out.returncode == 0 and out.stdout.strip(), (
    'No GPU found. Use Runtime -> Change runtime type -> T4 GPU, then rerun.')
print('GPU:', out.stdout.strip())

## 2. Storage (Google Drive or local)

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    HJEPA_HOME = '/content/drive/MyDrive/hjepa_home'
else:
    HJEPA_HOME = '/content/hjepa_home'
os.makedirs(HJEPA_HOME, exist_ok=True)

REPO = '/content/H-JEPA'
VENV = '/content/hjepa-venv'
PY = f'{VENV}/bin/python'

# Environment for every repo command. MUJOCO_GL / PYOPENGL_PLATFORM: headless MuJoCo
# rendering on the GPU (AntMaze, Cube). SDL_VIDEODRIVER: headless pygame (FourRoom, Push-T).
# MPLBACKEND: Colab sets it to its inline backend, which does not exist in the venv.
ENV = dict(os.environ, HJEPA_HOME=HJEPA_HOME, MUJOCO_GL='egl', PYOPENGL_PLATFORM='egl',
           SDL_VIDEODRIVER='dummy', MPLBACKEND='Agg', PYTHONUNBUFFERED='1', HYDRA_FULL_ERROR='1')
print('HJEPA_HOME =', HJEPA_HOME)

## 3. Get the code (pinned commit)

In [ ]:
if not os.path.isdir(f'{REPO}/.git'):
    run(f'git clone {REPO_URL} {REPO}')
run(f'git -C {REPO} checkout -q {REPO_COMMIT}')
head = subprocess.run(f'git -C {REPO} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
assert head == REPO_COMMIT, f'unexpected commit {head}'
print('code at', head)

## 4. Install dependencies (about 5 to 10 minutes)

The repository requires **Python 3.11 exactly** and pins `torch==2.7.1` and `mujoco==3.14.0`. Colab's own Python may be a different version, so the cell builds a separate Python 3.11 environment with `uv` and runs all repo code inside it.

- The PyPI `torch==2.7.1` wheel is the CUDA 12.6 build, which supports the T4.
- The evaluation runs in fp32, so the T4's missing bf16 support doesn't matter here.

In [ ]:
run('apt-get -qq update && apt-get -qq install -y libegl1 libgl1 libglu1-mesa > /dev/null', check=False)
run(f'{sys.executable} -m pip install -q uv')
if not os.path.exists(PY):
    run(f'uv venv --python 3.11 {VENV}')
run(f'uv pip install --python {PY} -e ".[train,env]"', cwd=REPO)
run(f'uv pip install --python {PY} "huggingface_hub>=0.34"')

## 5. Environment checks
Each check asserts, so a broken install stops here instead of failing later in the middle of an evaluation.

In [ ]:
CHECK_ENV = r'''
import os, sys, torch
assert sys.version_info[:2] == (3, 11), sys.version
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda)
assert torch.__version__.startswith('2.7.1'), torch.__version__
assert torch.cuda.is_available(), 'torch cannot see the GPU'
cap = torch.cuda.get_device_capability()
arch = f'sm_{cap[0]}{cap[1]}'
print('device', torch.cuda.get_device_name(), arch, '| torch archs', torch.cuda.get_arch_list())
assert arch in torch.cuda.get_arch_list(), f'this torch build has no kernels for {arch}'
x = torch.randn(512, 512, device='cuda', requires_grad=True)
(x @ x).sum().backward()
assert torch.isfinite(x.grad).all()
print('GPU matmul + backward: OK')

import matplotlib, lightning, hydra, sklearn, stable_worldmodel as swm, gymnasium as gym
print('matplotlib backend', matplotlib.get_backend())
print('stable_worldmodel imported from', swm.__file__)
import mujoco
print('mujoco', mujoco.__version__)
'''
run(f'{PY} -c "$CHECK_ENV"', env=dict(ENV, CHECK_ENV=CHECK_ENV), cwd=f'{REPO}/h_jepa')

## 6. Download checkpoints and evaluation tasks
Files come from the Hugging Face repos `jepa-world-models/h-jepa` (model and dataset). Files already on Drive are skipped. The task files are verified against the published SHA-256 sums.

In [ ]:
TASK_DIR = {'fourroom': 'fourroom', 'ant': 'ant', 'cube': 'ogbench', 'pusht': 'pusht'}
for env_name, model, seed in RUNS:
    assert env_name in TASK_DIR, env_name
    assert model in {'lewm', 'hjepa_l2', 'hjepa_l3', 'hjepa_l4', 'hwm_l2', 'hwm_l3', 'hwm_l4'}, model
    assert seed in (42, 43, 44), seed

ckpt_patterns = sorted({f'{e}/{e}_{m}/seed{s}/*' for e, m, s in RUNS})
task_patterns = sorted({f'eval_trajs/{TASK_DIR[e]}/*' for e, _, _ in RUNS}) + ['eval_trajs/SHA256SUMS']

DOWNLOAD = f'''
from huggingface_hub import snapshot_download
snapshot_download('jepa-world-models/h-jepa', repo_type='model',
                  local_dir='{HJEPA_HOME}/ckpts', allow_patterns={ckpt_patterns!r})
snapshot_download('jepa-world-models/h-jepa', repo_type='dataset',
                  local_dir='{HJEPA_HOME}', allow_patterns={task_patterns!r})
print('download done')
'''
run(f'{PY} -c "$DOWNLOAD"', env=dict(ENV, DOWNLOAD=DOWNLOAD))

# Verify task files, then link them where the eval configs look for them (h_jepa/assets/eval_trajs).
run('sha256sum -c --ignore-missing SHA256SUMS', cwd=f'{HJEPA_HOME}/eval_trajs')
assets = pathlib.Path(f'{REPO}/h_jepa/assets')
assets.mkdir(exist_ok=True)
link = assets / 'eval_trajs'
if link.is_symlink() or link.exists():
    assert link.is_symlink(), f'{link} exists and is not a symlink; remove it and rerun'
    link.unlink()
link.symlink_to(f'{HJEPA_HOME}/eval_trajs')

def ckpt_path(e, m, s):
    return f'{HJEPA_HOME}/ckpts/{e}/{e}_{m}/seed{s}/{e}_{m}_object.ckpt'

for e, m, s in RUNS:
    p = ckpt_path(e, m, s)
    assert os.path.isfile(p), f'missing {p}'
    assert os.path.isfile(os.path.join(os.path.dirname(p), 'normalizer.pt')), f'missing normalizer for {p}'
    print(f'OK  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')

## 7. Checkpoint checks
Each checkpoint is loaded the way `eval.py` loads it, then checked for:
- the expected depth and parameter count;
- finite weights;
- the stored latent-action statistics that the upper-level planners need.

In [ ]:
CHECK_CKPT = r'''
import os, sys, json, torch
for path, want_levels in json.loads(os.environ['CKPTS']):
    model = torch.load(path, map_location='cpu', weights_only=False)
    n_params = sum(p.numel() for p in model.parameters())
    levels = getattr(model, 'num_levels', 1)
    finite = all(torch.isfinite(p).all() for p in model.parameters())
    print(f'{os.path.basename(path)}: {type(model).__name__}, levels={levels}, '
          f'params={n_params/1e6:.2f}M, finite={finite}')
    assert finite, 'non-finite weights'
    assert levels == want_levels, f'expected {want_levels} levels, got {levels}'
    # Upper-level planners initialise macro-actions from each level's stored latent-action queue.
    for lvl in range(2, levels + 1):
        q = model.get_level(lvl).get_latent_action_queue()
        assert q is not None and q.ndim == 2, f'level {lvl} has no latent-action queue'
        print(f'  level {lvl}: latent-action queue {tuple(q.shape)}')
'''
def depth(model):
    return 1 if model == 'lewm' else int(model[-1])
ckpts = [(ckpt_path(e, m, s), depth(m)) for e, m, s in RUNS]
run(f'{PY} -c "$CHECK_CKPT"', cwd=f'{REPO}/h_jepa',
    env=dict(ENV, CHECK_CKPT=CHECK_CKPT, CKPTS=json.dumps(ckpts)))

## 8. Planning evaluation

This uses the repository's own command, the one `scripts/eval_depth.sh` runs:
`python eval.py --config-name <env>_<planner> seed=<seed> policy=<ckpt> output.dir=<dir>`.

- The planner matches the model: `flat` for LeWM, `lN` for an N-level model.
- Planner settings come unchanged from `h_jepa/config/eval/`.
- Each run uses the 50 fixed start/goal tasks.

The smoke test first runs `SMOKE_TASKS` tasks into a separate folder, to prove the full pipeline works and to time one task. The full evaluation then writes to `colab_eval_<planner>/` next to the checkpoint.

In [ ]:
def planner(model):
    return 'flat' if model == 'lewm' else model.split('_')[1]  # hjepa_l3 -> l3

def eval_cmd(e, m, s, out_dir, extra=''):
    return (f'{PY} eval.py --config-name {e}_{planner(m)} seed={s} '
            f'policy={ckpt_path(e, m, s)} output.dir={out_dir} {extra}')

def results_dir(e, m, s, out_dir):
    return os.path.join(os.path.dirname(ckpt_path(e, m, s)), out_dir)

if SMOKE_TASKS:
    for e, m, s in RUNS:
        out = f'colab_smoke{SMOKE_TASKS}_{planner(m)}'
        t0 = time.time()
        run(eval_cmd(e, m, s, out, f'eval.num_eval={SMOKE_TASKS}'), cwd=f'{REPO}/h_jepa', env=ENV)
        chunks = sorted(pathlib.Path(results_dir(e, m, s, out), 'chunks').glob('tasks_*.json'))
        assert len(chunks) == SMOKE_TASKS, f'expected {SMOKE_TASKS} task files, found {len(chunks)}'
        per_task = [json.loads(c.read_text())['evaluation_time'] for c in chunks]
        print(f'\nSMOKE OK {e}/{m}/seed{s}: {sum(per_task)/len(per_task):.0f} s per task -> '
              f'about {50*sum(per_task)/len(per_task)/3600:.1f} h for all 50 tasks '
              f'(wall time incl. model load {time.time()-t0:.0f} s)\n')

In [ ]:
if RUN_FULL_EVAL:
    for e, m, s in RUNS:
        run(eval_cmd(e, m, s, f'colab_eval_{planner(m)}'), cwd=f'{REPO}/h_jepa', env=ENV)

## 9. Compare with the paper

Paper values are mean ± SE over seeds 42, 43 and 44, from `docs/RESULTS.md` (Figure 6, bottom row).

One seed with 50 tasks is a noisy estimate: the binomial standard error is up to about 7 points. So this cell checks two things:
- **Per run:** whether the paper's mean lies inside the 95% Wilson interval of our success rate.
- **Over seeds:** when all three seeds of a model have been run, it also prints our mean ± SE next to the paper's.

In [ ]:
import math, yaml, statistics

PAPER = {  # success rate %, (mean, SE) over seeds 42/43/44
    'ant':      {'lewm': (18.0, 3.5), 'hjepa_l2': (39.3, 3.7), 'hjepa_l3': (73.3, 3.5), 'hjepa_l4': (67.3, 2.4),
                 'hwm_l2': (34.0, 1.2), 'hwm_l3': (38.7, 3.5), 'hwm_l4': (14.0, 2.0)},
    'fourroom': {'lewm': (40.7, 6.8), 'hjepa_l2': (81.3, 6.7), 'hjepa_l3': (96.0, 1.1), 'hjepa_l4': (88.0, 2.3),
                 'hwm_l2': (75.3, 3.3), 'hwm_l3': (99.3, 0.7), 'hwm_l4': (92.7, 4.4)},
    'cube':     {'lewm': (32.0, 6.1), 'hjepa_l2': (47.3, 2.7), 'hjepa_l3': (60.0, 5.0), 'hjepa_l4': (62.7, 2.4),
                 'hwm_l2': (46.7, 2.9), 'hwm_l3': (53.3, 5.2), 'hwm_l4': (34.0, 3.1)},
    'pusht':    {'lewm': (40.0, 2.3), 'hjepa_l2': (45.3, 1.3), 'hjepa_l3': (17.3, 3.5), 'hjepa_l4': (0.7, 0.7),
                 'hwm_l2': (42.0, 6.4), 'hwm_l3': (12.0, 2.0), 'hwm_l4': (0.0, 0.0)},
}

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return 100*(c-h), 100*(c+h)

rows, by_model = [], {}
for e, m, s in RUNS:
    f = pathlib.Path(results_dir(e, m, s, f'colab_eval_{planner(m)}'), 'metrics.yaml')
    if not f.exists():
        print(f'{e}/{m}/seed{s}: no metrics.yaml yet (full eval not finished)')
        continue
    met = yaml.safe_load(f.read_text())
    succ = met['episode_successes']
    n, k = len(succ), int(sum(succ))
    sr = met['success_rate']
    assert n == 50 and abs(sr - 100*k/n) < 1e-6, (n, sr, k)
    lo, hi = wilson(k, n)
    pm, pse = PAPER[e][m]
    rows.append((e, m, s, sr, lo, hi, pm, pse, lo <= pm <= hi, met['evaluation_time']/3600))
    by_model.setdefault((e, m), []).append(sr)

print(f"{'env':9} {'model':9} {'seed':4} {'ours %':>7} {'95% CI':>13} {'paper %':>12} {'consistent':>10} {'hours':>6}")
for e, m, s, sr, lo, hi, pm, pse, ok, hrs in rows:
    print(f'{e:9} {m:9} {s:<4} {sr:7.1f} {lo:6.1f}-{hi:5.1f} {pm:6.1f} ± {pse:3.1f} {"yes" if ok else "NO":>10} {hrs:6.2f}')
for (e, m), srs in by_model.items():
    if len(srs) >= 2:
        se = statistics.stdev(srs) / math.sqrt(len(srs))
        print(f'{e}/{m}: ours {statistics.mean(srs):.1f} ± {se:.1f} over {len(srs)} seeds | '
              f'paper {PAPER[e][m][0]} ± {PAPER[e][m][1]}')

## Notes

- **Where results live:** `$HJEPA_HOME/ckpts/<env>/<env>_<model>/seed<seed>/colab_eval_<planner>/`. That folder holds `metrics.yaml`, one JSON per task under `chunks/`, `planning_compute.json` (planner iterations and wall clock per solve) and `eval_config.yaml`.
- **Resuming:** rerun all cells. Finished tasks are skipped.
- **Changing planner settings** (for example `solver.solvers.level1.num_samples=100`): add them to `eval_cmd` and use a new `output.dir`. Otherwise old task results are reused.
- **AntMaze and Cube render with MuJoCo on the GPU (EGL).** If they fail with an EGL or OpenGL error, set `MUJOCO_GL='osmesa'` and `PYOPENGL_PLATFORM='osmesa'` in cell 2, and run `apt-get install -y libosmesa6`. Rendering on the CPU is slower but works.